 ## - Architecture Medallion et Parquet
Objectif
Mettre en place l'architecture Data Lake Medallion du projet AgriLens.

Les couches visées sont :

## Bronze
## Silver
## Gold
Les tables seront sérialisées au format Parquet afin d'améliorer les performances de lecture, la compression des données et la compatibilité avec les futurs traitements PySpark.

### Livrables
Bronze :

inventory.parquet
Silver :

inventory_clean.parquet
Gold :

reporté à une étape ultérieure en attente du Class Mapping.

In [1]:
# but c'est de structure la donnée dans des couche de fichier par niveau de maturité

In [2]:
import pandas as pd

In [3]:
from pathlib import Path

Path("../../01_data/lake/bronze").exists()

True

In [4]:
Path("../../01_data/lake/silver").exists()

True

In [5]:
Path("../../01_data/lake/gold").exists()

True

In [6]:
inventory_bronze = pd.read_csv(
    "../../01_data/metadata/inventory.csv"
)

inventory_silver = pd.read_csv(
    "../../01_data/metadata/inventory_v3.csv"
)

In [7]:
print("Bronze :", inventory_bronze.shape)
print("Silver :", inventory_silver.shape)

Bronze : (26402, 5)
Silver : (26397, 20)


In [8]:
# Validation des sources Medallion
#
# Bronze :
#  inventory.csv
#  26402 images
#  5 colonnes
#  représentation brute du dataset
#
# Silver :
#  inventory_v3.csv
#  26397 images
#  20 colonnes
#  représentation nettoyée et enrichie
#
# Conclusion :
# Les sources destinées aux couches
# Bronze et Silver sont correctement
# identifiées et disponibles.


## Vérification du support Parquet

Les fichiers Bronze et Silver seront sérialisés au format Parquet avec  PyArrow. de pandas


In [9]:
import pyarrow

print(pyarrow.__version__)

24.0.0


## Couche Bronze
### Objectif :

Créer la représentation Parquet de l'inventaire brut du projet AgriLens.

Cette couche constitue le miroir des données avant les opérations d'enrichissement et de nettoyage.

Source :

inventory.csv
Destination :

01_data/lake/bronze/inventory.parquet

In [10]:
import pandas as pd

pd.__version__

'3.0.6'

In [11]:
import pyarrow as pa

pa.__version__

'24.0.0'

OK


In [15]:
inventory_bronze.to_parquet(
    "../../01_data/lake/bronze/inventory.parquet",
    engine="pyarrow",
    index=False
)

In [16]:
from pathlib import Path

Path(
    "../../01_data/lake/bronze/inventory.parquet"
).exists()

True

In [17]:
bronze_check = pd.read_parquet(
    "../../01_data/lake/bronze/inventory.parquet"
)

bronze_check.shape

(26402, 5)

## Couche Silver
### Objectif :

Créer la représentation Parquet du dataset nettoyé et enrichi du projet AgriLens.

Cette couche contient :

les métadonnées géométriques ;
les métadonnées photométriques ;
les indicateurs de qualité ;
les informations de détection de doublons ;
les informations de contrôle du leakage.
Source :

## inventory_v3.csv

Destination :

01_data/lake/silver/inventory_clean.parquet

In [18]:
inventory_silver.to_parquet(
    "../../01_data/lake/silver/inventory_clean.parquet",
    index=False
)

In [19]:
from pathlib import Path

Path(
    "../../01_data/lake/silver/inventory_clean.parquet"
).exists()

True

In [20]:
silver_check = pd.read_parquet(
    "../../01_data/lake/silver/inventory_clean.parquet"
)

silver_check.shape

(26397, 20)

In [21]:
# Statut de la couche Gold
#
# La couche Gold n'est pas créée à ce stade.
#
# Motif :
#
# Le guide impose que la table Gold soit :
#
# - raccordée au Split Manifest ;
# - indexée par un Class Mapping immuable.
#
# Le Class Mapping n'ayant pas encore été produit,
# la génération de gold_dataset.parquet est reportée.
#
# Décision :
#
#  Bronze validée
# Silver validée
# ⏳ Gold reportée
#
# La couche Gold sera construite lors de la mise
# en place du Class Mapping et de la préparation
# du DataLoader PyTorch.